# Basket Recommender: Run and Test in Google Colab

This notebook checks that the basket recommender works from start to finish. Run the cells **in order, top to bottom** (Runtime ▸ Run all also works, but the chat cell in step 8 waits for you to type). No GPU is needed.

| Step | What it does |
|---|---|
| 1 | Loads the project (from GitHub, or from the ZIP file you upload) |
| 2 | Installs the libraries it needs |
| 3 | Runs the original mlxtend example the project is based on |
| 4 | Runs the 13 automated tests |
| 5 | Runs the evaluation and draws the results chart |
| 6 | Shows recommendations for a few sample baskets |
| 7 | Replays a scripted chat session |
| 8 | Lets you chat with the assistant yourself |
| 9 | Opens a point-and-click interface inside the notebook |
| 10 | *(Optional)* Launches the Streamlit web app |

## 1. Load the project

**Option A, from GitHub:** paste your repository URL between the quotes, for example `https://github.com/your-username/basket-recommender.git`. Private repositories need a token, so Option B is easier for those.

**Option B, from the ZIP:** leave `REPO_URL` empty. When you run the cell, a **Choose Files** button appears; select `basket-recommender.zip`.

In [ ]:
REPO_URL = ""  # e.g. "https://github.com/your-username/basket-recommender.git"

import os, shutil, subprocess, sys, zipfile, glob

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

BASE = "/content" if IN_COLAB else os.path.abspath(".")
PROJECT_DIR = os.path.join(BASE, "basket-recommender")

if os.path.exists(os.path.join(PROJECT_DIR, "recommender")):
    print("Project already loaded.")
elif REPO_URL:
    subprocess.run(["git", "clone", "-q", REPO_URL, PROJECT_DIR], check=True)
    print("Cloned", REPO_URL)
elif IN_COLAB:
    from google.colab import files
    uploaded = files.upload()  # pick basket-recommender.zip
    zip_name = next(name for name in uploaded if name.endswith(".zip"))
    with zipfile.ZipFile(zip_name) as z:
        z.extractall(BASE)
    # Accept a ZIP whose top folder has a different name (e.g. GitHub's "basket-recommender-main")
    if not os.path.exists(PROJECT_DIR):
        found = glob.glob(os.path.join(BASE, "*", "recommender", "engine.py"))
        if not found:
            raise FileNotFoundError("Couldn't find the project inside the ZIP.")
        os.rename(os.path.dirname(os.path.dirname(found[0])), PROJECT_DIR)
    print("Unzipped", zip_name)
else:
    raise FileNotFoundError("Set REPO_URL or put the basket-recommender folder next to this notebook.")

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
print("Working in", os.getcwd())
print(sorted(os.listdir(".")))

## 2. Install the libraries

Colab already has pandas, pytest, matplotlib and ipywidgets. This makes sure mlxtend is recent enough (0.23.2 or later).

In [ ]:
%pip install -q "mlxtend>=0.23.2" tabulate
import mlxtend, pandas
print("Python", sys.version.split()[0], "| mlxtend", mlxtend.__version__, "| pandas", pandas.__version__)

## 3. Run the original example (the foundation)

This is Example 1 from the mlxtend user guide by Sebastian Raschka, reproduced unchanged. It ends with a table of rules and recommends nothing; everything after this step is the project's own work.

In [ ]:
!python original_example/mlxtend_baseline.py

## 4. Run the automated tests

You should see **13 passed**. The tests check that the from-scratch Apriori gives exactly the same results as mlxtend, and that recommendations, explanations, exclusions, diversity, the empty-basket fallback and the chat parser behave as documented.

In [ ]:
!python -m pytest -q -p no:cacheprovider

## 5. Run the evaluation

This trains on 80% of the baskets, hides one item from each test basket, and checks whether the hidden item appears in the top 5 suggestions. The expected hit rate is **0.728** for the rule-based system versus **0.323** for the popularity baseline. The data are generated from a fixed seed, so your numbers should match exactly.

In [ ]:
!python evaluate.py

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

results = pd.read_csv("results/systems.csv").iloc[::-1]
colors = ["#8a8984" if s == "Popularity baseline" else "#2a78d6" for s in results.System]
fig, ax = plt.subplots(figsize=(7, 3.2))
bars = ax.barh(results.System, results["hit_rate@5"], color=colors, height=0.55)
for bar, value in zip(bars, results["hit_rate@5"]):
    ax.text(value + 0.01, bar.get_y() + bar.get_height() / 2, f"{value:.3f}", va="center", fontsize=9)
ax.set_xlim(0, 0.85)
ax.set_xlabel("Hit rate@5")
ax.set_title("Hit rate@5 by system version (reproduced in Colab)", fontsize=11)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.xaxis.grid(True, color="#ecebe7")
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## 6. Try the recommender from Python

Each suggestion explains which rule produced it. The empty basket shows the cold-start fallback to popular items.

In [ ]:
from recommender import BasketRecommender, load_catalog, load_transactions

model = BasketRecommender().fit(load_transactions(), load_catalog())
print(f"Learned {len(model.rules)} rules from {len(load_transactions())} baskets.\n")

for basket in (["Pasta"], ["Tortillas", "Ground Beef"], ["Diapers"], []):
    print("Basket:", basket or "(empty)")
    for rec in model.recommend(basket, k=4):
        print("   ", rec.explain())
    print()

print("Same taco basket, with diversity 0.5 and no dairy:")
for rec in model.recommend(["Tortillas", "Ground Beef"], k=4, exclude_categories=["Dairy & Eggs"], diversity=0.5):
    print("   ", rec.explain())

## 7. Replay a scripted chat session

This sends a fixed set of messages to the conversational assistant (`app_cli.py`), including a typo (“tortilas”) that it corrects.

In [ ]:
from app_cli import build_assistant

bot = build_assistant()
for message in ["I have pasta and parmesan", "why pasta sauce", "avoid dairy",
                "add tortilas, ground beef", "remove pasta", "show"]:
    print(">", message)
    print(bot.handle(message))
    print()

## 8. Chat with the assistant yourself

Run the cell and type in the box that appears. Try plain sentences such as *I've got cereal and bananas*, or commands such as `avoid meat`, `why salsa`, `remove cereal`, `diversity 0.5` or `help`.

Type **quit** to finish. The cell keeps running until you do, so stop it before running later cells.

In [ ]:
from app_cli import build_assistant

chat = build_assistant()
if IN_COLAB:
    print("Basket assistant ready. Type 'help' for commands or 'quit' to stop.")
    while True:
        message = input("> ")
        reply = chat.handle(message)
        if reply:
            print(reply)
        if message.strip().lower() in {"quit", "exit", "bye"}:
            break
else:
    print("Interactive chat runs in Colab; outside it, see the scripted session in step 7.")

## 9. Point-and-click interface

This gives you the features of the Streamlit app (`app_gui.py`) inside the notebook. Pick items for your basket and the suggestions update as you change them.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

gui_model = BasketRecommender().fit(load_transactions(), load_catalog())
categories = sorted(set(gui_model.categories.values()))
layout = widgets.Layout(width="320px", height="180px")

basket_w = widgets.SelectMultiple(options=gui_model.items, description="Basket", layout=layout)
avoid_w = widgets.SelectMultiple(options=categories, description="Avoid", layout=widgets.Layout(width="320px", height="180px"))
k_w = widgets.IntSlider(value=5, min=1, max=10, description="Suggestions")
div_w = widgets.FloatSlider(value=0.0, min=0.0, max=0.9, step=0.1, description="Diversity")
out = widgets.Output()

def refresh(_=None):
    with out:
        out.clear_output()
        recs = gui_model.recommend(list(basket_w.value), k_w.value,
                                   exclude_categories=list(avoid_w.value), diversity=div_w.value)
        print("Basket:", ", ".join(basket_w.value) or "(empty)")
        print("Hold Ctrl (Cmd on a Mac) to pick several items.\n")
        for n, rec in enumerate(recs, 1):
            print(f"{n}. {rec.explain()}")
        if not recs:
            print("Nothing to suggest with these filters.")

for w in (basket_w, avoid_w, k_w, div_w):
    w.observe(refresh, names="value")
refresh()
display(widgets.VBox([widgets.HBox([basket_w, avoid_w]), widgets.HBox([k_w, div_w]), out]))

## 10. *(Optional)* Launch the Streamlit web app

This starts `app_gui.py` and opens it through Colab's built-in port forwarding, in a new browser tab. It installs Streamlit first, which takes about a minute.

If the page shows an error or stays blank, your Colab setup is blocking the connection. The interface in step 9 has the same features, and `streamlit run app_gui.py` works normally on your own computer.

In [ ]:
if IN_COLAB:
    %pip install -q streamlit
    import subprocess, time
    from google.colab import output

    log = open("/content/streamlit.log", "w")
    subprocess.Popen(["streamlit", "run", "app_gui.py", "--server.port", "8501", "--server.headless", "true",
                      "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],
                     stdout=log, stderr=log)
    time.sleep(8)
    output.serve_kernel_port_as_window(8501)
else:
    print("Skipped: this step is only for Colab. Locally, run: streamlit run app_gui.py")

## Done

If steps 3–9 ran without errors, the tests showed **13 passed**, and the evaluation reported a hit rate of **0.728**, the system is working. The report's numbers come from this same code and data.